# Full End-to-End Pipeline Demo

This notebook demonstrates the complete daily trading pipeline:

```
Market Data (yfinance)
  → Feature Engineering (returns, momentum, vol_ratio)
    → Alpha Scoring (20d momentum per ticker)
      → Top-N Selection
        → Multi-Agent Analysis (LangGraph)
          → Signal Generation
            → Portfolio Optimizer
              → Pre-Trade Risk Controls
                → PaperTrader Execution
                  → Portfolio Snapshot
```


In [ ]:
import sys
from pathlib import Path

# Add project root to path
repo_root = Path('.').resolve().parent
sys.path.insert(0, str(repo_root))
print(f'Project root: {repo_root}')


In [ ]:
# Step 1: Fetch OHLCV data
import pandas as pd
import numpy as np
from src.utils.config import get_config
from src.core.data_pipeline import DataPipeline

config = get_config()
pipeline = DataPipeline(config)

UNIVERSE = ['SPY', 'AAPL', 'MSFT', 'GOOGL', 'AMZN', 'NVDA', 'TSLA', 'JPM', 'V', 'UNH']

try:
    df = pipeline.yfinance_fallback(tickers=UNIVERSE, start='2023-01-01', end='2023-03-31')
    print(f'Fetched {len(df)} rows for {df["ticker"].nunique()} tickers.')
except Exception as e:
    print(f'Data fetch failed ({e}), using synthetic data.')
    dates = pd.bdate_range('2023-01-01', periods=60)
    frames = []
    for ticker in UNIVERSE:
        np.random.seed(hash(ticker) % 2**31)
        prices = 100 * np.cumprod(1 + np.random.normal(0.0005, 0.02, len(dates)))
        frame = pd.DataFrame({
            'close': prices, 'open': prices * 0.999, 'high': prices * 1.01,
            'low': prices * 0.99, 'volume': np.random.randint(1_000_000, 10_000_000, len(dates)),
            'ticker': ticker,
        }, index=dates)
        frames.append(frame)
    df = pd.concat(frames)
    df.index.name = 'date'
    print(f'Synthetic data: {len(df)} rows for {len(UNIVERSE)} tickers.')


In [ ]:
# Step 2: Initialize PaperTrader
from src.execution.paper_trader import PaperTrader

paper_trader = PaperTrader(
    initial_cash=100_000.0,
    slippage_bps=5,
    commission_per_share=0.005,
)
print(f'PaperTrader initialized: cash={paper_trader.portfolio.cash:,.2f}')
print(f'Initial NAV: {paper_trader.portfolio.nav:,.2f}')


In [ ]:
# Step 3: Compute alpha scores and select top-5 tickers
alpha_scores = {}
for ticker in UNIVERSE:
    ticker_df = df[df['ticker'] == ticker].sort_index()
    if ticker_df.empty:
        alpha_scores[ticker] = 0.0
        continue
    mom20 = ticker_df['close'].pct_change(20)
    alpha_scores[ticker] = float(mom20.mean()) if not mom20.isna().all() else 0.0

top_tickers = sorted(alpha_scores, key=lambda t: alpha_scores[t], reverse=True)[:5]
print('Alpha scores (top 5):')
for t in top_tickers:
    print(f'  {t:8s}: {alpha_scores[t]:.4f}')


In [ ]:
# Step 4: Multi-agent analysis for top tickers
from src.agents.graph import analyze_ticker
from src.utils.schemas import AnalysisResult, Decision, Signal, SignalDirection

results = {}
for ticker in top_tickers:
    score = alpha_scores[ticker]
    qlib_context = f'Alpha score: {score:.4f}'
    print(f'Analyzing {ticker}...')
    try:
        result = analyze_ticker(ticker, qlib_context=qlib_context)
        results[ticker] = result
        print(f'  {ticker}: {result.decision.value} (confidence={result.confidence:.1f}%)')
    except Exception as e:
        print(f'  {ticker}: analysis failed ({e})')
        results[ticker] = AnalysisResult(
            ticker=ticker,
            decision=Decision.HOLD,
            confidence=40.0,
            reasoning=f'Stub result: {str(e)[:80]}',
        )


In [ ]:
# Step 5: Generate signals, translate to orders, execute via PaperTrader
from src.execution.signal_translator import signals_to_orders
from src.execution.risk_controls import check_order

# Build signals from agent results
signals = []
for ticker, result in results.items():
    decision_val = result.decision.value
    direction = SignalDirection.LONG if decision_val in ('STRONG_BUY', 'BUY') else SignalDirection.FLAT
    signals.append(Signal(
        ticker=ticker,
        direction=direction,
        strength=min(result.confidence / 100.0, 1.0),
        source='notebook_pipeline',
    ))

# Current market prices (last close)
market_prices = {}
for ticker in top_tickers:
    ticker_rows = df[df['ticker'] == ticker]
    if not ticker_rows.empty:
        market_prices[ticker] = float(ticker_rows['close'].iloc[-1])
    else:
        market_prices[ticker] = 100.0

# Translate signals to orders
portfolio = paper_trader.portfolio
orders = signals_to_orders(signals, portfolio, market_prices)
print(f'Generated {len(orders)} orders from {len(signals)} signals.')

# Run risk checks and execute passing orders
orders_placed = 0
for order in orders:
    risk_result = check_order(order, portfolio, market_prices, config.risk)
    if not risk_result.passed:
        print(f'  BLOCKED {order.ticker}: {risk_result.failed_checks}')
        continue
    try:
        filled = paper_trader.execute_order(order, market_prices)
        if filled is not None:
            orders_placed += 1
            print(f'  FILLED {order.side.value} {order.ticker}: qty={order.quantity:.0f} @ {filled.fill_price:.2f}')
    except Exception as e:
        print(f'  ERROR {order.ticker}: {e}')

print(f'\nOrders placed: {orders_placed}')


In [ ]:
# Step 6: Take portfolio snapshot and display results
paper_trader.snapshot(market_prices)
final = paper_trader.portfolio

print('\nPortfolio Summary:')
print(f'{"Ticker":10s} {"Qty":>8s} {"Price":>10s} {"Weight%":>8s}')
print('-' * 40)
for ticker, pos in final.positions.items():
    price = market_prices.get(ticker, pos.current_price)
    print(f'{ticker:10s} {pos.quantity:>8.0f} {price:>10.2f} {pos.weight_pct:>8.1f}%')
print('-' * 40)
print(f'{"CASH":10s} {"":>8s} {"":>10s} ${final.cash:>10,.2f}')
print(f'{"NAV":10s} {"":>8s} {"":>10s} ${final.nav:>10,.2f}')


## Summary

This notebook demonstrated the full end-to-end pipeline:

1. **Data** — `DataPipeline.yfinance_fallback()` fetches OHLCV for the universe
2. **Alpha** — 20-day momentum score per ticker, top-5 selected
3. **Analysis** — `analyze_ticker()` runs the multi-agent LangGraph pipeline
4. **Signals** — `signals_to_orders()` converts agent decisions to order objects
5. **Risk** — `check_order()` enforces pre-trade risk limits (6 checks)
6. **Execution** — `PaperTrader.execute_order()` simulates fills with slippage
7. **Snapshot** — `PaperTrader.snapshot()` marks positions to market

To run the pipeline continuously:
```bash
python scripts/run_pipeline.py run --mode paper-loop
```

To analyze a single ticker:
```bash
python scripts/run_agents.py analyze AAPL --output rich
```
